# Algorithmic Trading Strategy: walkthrough

A step-by-step tour of the engine: load data, form a signal, backtest it with an execution lag and trading costs, and test whether the result is distinguishable from luck.

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
from algo_trading import load_data, sma_crossover, xs_momentum, buy_and_hold, run_backtest, Costs, performance, SECTORS
from algo_trading import stats_tests as T
px, rf, info = load_data()
info['source'], info['start'], info['end']

## 1. Point-in-time universe
Sector funds that launched later have no price before launch, so they cannot be selected early. This avoids survivorship bias from building today's universe into the past.

In [ ]:
pd.Series(info['first_valid'])

## 2. A signal is a target weight decided at the close
The SMA 50/200 rule: hold SPY when the 50-day average is above the 200-day average, otherwise T-bills.

In [ ]:
sig = sma_crossover(px['SPY'], 50, 200)
sig.dropna().tail()

## 3. Backtest with a one-day execution lag and 10 bps per trade
A signal formed at Monday's close trades at Tuesday's close; trading costs come out of equity on the trade date.

In [ ]:
costs = Costs(commission_bps=5, slippage_bps=5)
sma = run_backtest(px, sig, rf, costs, lag=1)
spy = run_backtest(px, buy_and_hold(px[['SPY']]), rf, costs, lag=1)
start = sig.dropna().index[0]
pd.DataFrame({'SMA 50/200': performance(sma.returns.loc[start:], rf, spy.returns.loc[start:]),
              'SPY': performance(spy.returns.loc[start:], rf, spy.returns.loc[start:])}).round(3)

## 4. Look-ahead check
If we scramble every price after a cut-off date, nothing before the cut-off may change.

In [ ]:
cut = px.index[3000]
shocked = px.copy(); shocked.loc[shocked.index > cut] *= 1.5
alt = run_backtest(shocked, sma_crossover(shocked['SPY'], 50, 200), rf, costs)
(alt.returns.loc[:cut] == sma.returns.loc[:cut]).all()

## 5. Is the Sharpe improvement real?

In [ ]:
r, b = sma.returns.loc[start:], spy.returns.loc[start:]
bs = T.bootstrap_sharpe_diff(r, b, rf, n_boot=500)
{'PSR': T.probabilistic_sharpe(r, rf), 'Sharpe diff': bs['observed'], 'CI': (bs['ci_low'], bs['ci_high']), 'P(<=0)': bs['p_le_zero']}

## 6. Sector momentum
Hold the 3 sectors with the best 12-1 month return, rebalanced monthly; a sector with negative momentum is replaced by cash.

In [ ]:
xs = run_backtest(px, xs_momentum(px[[s for s in SECTORS if s in px]]), rf, costs)
pd.Series(performance(xs.returns.loc[start:], rf, b)).round(3)